# Improved CAM Localization

Stage 13 evaluates one separate Grad-CAM++ localization method using the frozen Phase 1 CNN. This notebook reads saved Stage 13 outputs; it does not train a model or rerun either evaluation.

## 1. Motivation

Stage 12 identified **spatial imprecision** as the dominant failure mode: activations were diffuse or misplaced, predicted boxes were larger than ground-truth boxes, and only 0.33% of baseline images achieved IoU >= 0.5. The threshold sweep's best mean IoU (0.087996 at 0.30) did not materially improve that high-IoU rate. The Stage 12 example panels also show salient activation outside annotated lung opacities.

## 2. Phase 1 Baseline

The official Phase 1 threshold-0.50 result is unchanged: mean IoU 0.080087, median IoU 0.062605, IoU > 0 on 80.70%, and IoU >= 0.5 on 0.33%, over the fixed 601 Pneumonia-positive test images. The Stage 8 CNN checkpoint, preprocessing statistics, split, box post-processing, and matching strategy are reused.

## 3. Proposed Improvement

Use Grad-CAM++ in place of classifier-weighted CAM to form the localization map. This is a single map-generation change intended to better weight class-specific spatial evidence. The CNN remains frozen; bounding boxes are never used to generate maps, set thresholds, choose regions, or fit parameters. The inclusive normalized-map threshold stays at 0.50 to avoid tuning the test set.

## 4. Method

```text
Image
  ↓
Frozen Stage 8 CNN
  ↓
Pneumonia-class Grad-CAM++ (per-image normalized, 128 × 128)
  ↓
Threshold at 0.50; iterative 8-connected regions
  ↓
Existing two-sigma region filter and native-coordinate boxes
  ↓
Ground-truth boxes enter only for final evaluation
```

The Stage 8 architecture has a linear classifier after global average pooling. The Grad-CAM++ implementation uses the corresponding closed-form first, second, and third derivatives of the exponential class score with respect to the final feature maps. Epsilon defaults to 1e-8 and is configurable with `--epsilon`; batches contain at most 16 images during full evaluation.

In [ ]:
import csv
import json
from pathlib import Path
from IPython.display import Image as DisplayImage, Markdown, display

ROOT = Path.cwd()
while not (ROOT / "results" / "phase2_improved_cam_results.json").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "results" / "phase2_improved_cam_results.json").is_file():
    raise FileNotFoundError("Could not locate saved Stage 13 results")

with (ROOT / "results" / "phase2_improved_cam_results.json").open(encoding="utf-8") as stream:
    results = json.load(stream)
with (ROOT / "results" / "phase2_baseline_vs_improved.csv").open(newline="", encoding="utf-8") as stream:
    comparison = list(csv.DictReader(stream))
with (ROOT / "results" / "phase2_improved_examples" / "examples.csv").open(newline="", encoding="utf-8") as stream:
    examples = list(csv.DictReader(stream))
print(f"Loaded evaluation for {results['test_sample_count']} fixed-test images and {len(examples)} comparison examples.")

## 5. Smoke Test

A deterministic 12-image smoke test passed: CAM generation, activation masks, connected components, finite values, in-bounds boxes, and visualization/source-image dimensional alignment were verified before full evaluation.

In [ ]:
smoke_preview = ROOT / "results" / "phase2_improved_examples" / "smoke_test_alignment.png"
display(DisplayImage(filename=str(smoke_preview)))

## 6. Full Test Evaluation

One full evaluation used the same 601 fixed-split Pneumonia-positive test images as Phase 1. No model weights, data splits, or thresholds were tuned.

In [ ]:
metric_rows = [
    ("Images evaluated", results["test_sample_count"]),
    ("Mean IoU", f"{results['mean_iou']:.6f}"),
    ("Median IoU", f"{results['median_iou']:.6f}"),
    ("IoU > 0", f"{results['percentage_iou_gt_0']:.2f}%"),
    ("IoU >= 0.5", f"{results['percentage_iou_ge_0_5']:.2f}%"),
]
display(Markdown("| Metric | Result |\n|---|---:|\n" + "\n".join(f"| {name} | {value} |" for name, value in metric_rows)))

## 7. Baseline vs Improved

The comparison uses the same fixed test set and the Phase 1 greedy one-to-one box IoU matching strategy.

In [ ]:
columns = ("Method", "Mean IoU", "Median IoU", "IoU > 0", "IoU >= 0.5")
lines = ["| " + " | ".join(columns) + " |", "|---|---:|---:|---:|---:|"]
for row in comparison:
    lines.append(
        f"| {row['Method']} | {float(row['Mean IoU']):.6f} | "
        f"{float(row['Median IoU']):.6f} | {float(row['IoU > 0']):.2f}% | "
        f"{float(row['IoU >= 0.5']):.2f}% |"
    )
display(Markdown("\n".join(lines)))
print(f"Absolute mean-IoU change: {results['absolute_improvement']:+.6f}")
print(f"Relative mean-IoU change: {results['relative_improvement_percent']:+.2f}%")

## 8. Qualitative Comparison

Ten deterministic examples include improved, unchanged, and worsened image-level IoU cases. Selection includes the best observed per-image gain, an unchanged case, the largest decrease, and further cases spanning the distribution of per-image changes; examples are not limited to successes.

In [ ]:
for case_type in ("improved", "unchanged", "worsened"):
    row = next((item for item in examples if item["case_type"] == case_type), None)
    if row is not None:
        print(f"{case_type}: baseline IoU {float(row['baseline_iou']):.4f}; improved IoU {float(row['improved_iou']):.4f}")
        display(DisplayImage(filename=str(ROOT / row["visualization"])))

## 9. Discussion

Grad-CAM++ improved the aggregate test mean IoU from 0.080087 to 0.090277 (+0.010190; +12.72%) and raised median IoU and the fraction with any overlap. It also exceeded Stage 12's best fixed-threshold CAM mean (0.087996). However, IoU >= 0.5 remained 0.33%, unchanged from Phase 1, and some visualized cases worsened. Therefore this experiment demonstrates a modest aggregate improvement on this fixed test evaluation, not universal localization improvement or out-of-sample generalization. The Phase 1 implementation and baseline result were not modified.